# 01A - Sequence Memory Theory

**Guiding question:** how can the first piano note still help with a guess twenty steps later?

> **What you already know:** a network makes a guess, loss checks it, backpropagation calculates feedback, and the optimizer changes weights.
> **The gap this chapter closes:** keeping an earlier clue available while new notes keep arriving.
> **What you'll have by the end:** a running notecard, a measured fading-feedback failure, and an LSTM route that can keep useful evidence longer.
> **What this chapter is not:** a trained music model or a PyTorch API tour. The practical batch contracts come in [01B's notebook](01B-sequence-memory-pytorch-lab.ipynb).

Stay with one short piano phrase: **D3 A3 D4**, then the same three notes again.
The numbers in these names tell a musician which octave to play; they aren't
model inputs yet. A listener hears a different tune if we reverse the notes.
Can a network keep that distinction, and can a later mistake teach it which
earlier clue mattered?

| Complaint | What we'll make visible |
|---|---|
| Counting notes loses their order | Same notes, reversed phrase, identical counts |
| Keeping every earlier note is awkward | One reused reader and a small carried notecard |
| The card exists, but feedback to old clues fades | A controlled route checked with autograd |
| We need to keep old evidence without overwriting it | Three LSTM controls: keep, write, show |
| Better routes don't guarantee useful learning | What the toy proves, and what 01B/01C still need to test |

The [handwritten notes](01A-sequence-memory-theory.md) keep the same complaint chain.


In [ ]:
# -- Setup: deterministic evidence, no training framework required ----------------
import numpy as np
import torch
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display

np.random.seed(42)
torch.manual_seed(42)
plt.rcParams.update({"figure.facecolor": "#fffaf1", "axes.facecolor": "#fffaf1"})

melody = "D3 A3 D4 D3 A3 D4".split()
print("Running example:", " ".join(melody))
print("Question: can an early note still matter when the phrase repeats?")


## 1 - Same notes, different order

Suppose we just count how often each note appears. **D3 A3 D4** and
**D4 A3 D3** each contain one of every note. A count card cannot tell
which tune we played. That information was thrown away before the model saw it.

![D3 A3 D4 and D4 A3 D3 occupy different ordered seats but produce identical note-count cards.](images/01a-01-order-is-not-a-bag.png)

*Read the note cards from left to right. Then compare the count cards on the right.
They match even though the tunes do not.*

Keep a separate seat for each position instead. In a batch, you now have:

- **Which phrase?** The batch axis keeps independent examples separate.
- **Which position in that phrase?** The time axis keeps their order.
- **What do we know at that position?** The feature axis holds that note's numbers.

A dense network with separate ordered input seats can distinguish order too.
The problem is **discarding** order, not dense layers being unable to use it.
Recurrence will add something else: one update rule we can reuse as notes arrive.

**Predict:** reverse our six-note phrase. Should its ordered seats change?
Should its note counts change? The next cell checks both.


In [ ]:
# -- Shape contract: preserve order instead of collapsing it ----------------------
vocab = sorted(set(melody))
token_to_id = {token: index for index, token in enumerate(vocab)}
ids = torch.tensor([[token_to_id[token] for token in melody]], dtype=torch.long)
# Keep identity separate from order: IDs [B,T] -> indicators [B,T,V], one vocabulary axis per position.
one_hot = torch.nn.functional.one_hot(ids, num_classes=len(vocab)).float()

# Sum over time: [B,T,V] -> [B,V] counts. Flipping T first changes order, not these totals.
forward_counts = one_hot.sum(dim=1)
reversed_counts = one_hot.flip(dims=(1,)).sum(dim=1)
same_bag = torch.equal(forward_counts, reversed_counts)
assert not torch.equal(ids, ids.flip(dims=(1,)))

print(f"Token IDs shape: {tuple(ids.shape)} = (batch, time)")
print(f"One-hot shape:   {tuple(one_hot.shape)} = (batch, time, feature)")
print(f"Forward and reversed bags equal: {same_bag}")
print("Takeaway: counts preserve membership; the time axis preserves order.")
assert same_bag


### The smallest useful notecard

Keeping ordered seats works for a fixed window. What if the phrase keeps going?
Instead of asking for a new input seat forever, let a reader carry a small
notecard. At each step it reads **the current note and the previous card**,
then writes a fresh card. That is a **recurrent neural network**, or RNN.

![D3, A3, and D4 each enter the same update rule, with the newly written notecard passed into the next use.](images/01a-02-one-reader-changing-card.png)

*The three reader boxes are three uses of the same weights, not three
independently trained readers. Only the carried card changes from step to step.*

Our card is a short row of numbers, not English notes or a saved transcript.
For a tiny arithmetic close-up, give the first three notes these **hand-picked**
input rows: `D3 -> [1, 0]`, `A3 -> [0, 1]`, `D4 -> [1, 1]`.
They are teaching values, not learned musical meanings.

The code mixes the current row with the old card, then uses `tanh` to keep
each new card value between -1 and 1. We start the phrase with a blank card.
The input-mixing and old-card-mixing weights stay the same at every step.

**Predict:** replay the notes backward from a fresh blank card. Must the
final card match? Unlike counting, the update reads a different history.


In [ ]:
# -- Manual recurrence: one cell reused through time ------------------------------
x_steps = torch.tensor([[1.0, 0.0], [0.0, 1.0], [1.0, 1.0]])
W_x = torch.tensor([[0.7, -0.2], [0.3, 0.8]])
W_h = torch.tensor([[0.6, 0.1], [-0.1, 0.5]])
state = torch.zeros(2)
states = []

for step, x_t in enumerate(x_steps):
    # Each [2,2] @ [2] -> [2] maps input or old memory into the same two state slots before adding.
    state = torch.tanh(W_x @ x_t + W_h @ state)
    states.append(state.clone())
    print(f"{melody[step]}: new card = {[round(value, 3) for value in state.tolist()]}")

reverse_state = torch.zeros(2)
for x_t in x_steps.flip(dims=(0,)):
    reverse_state = torch.tanh(W_x @ x_t + W_h @ reverse_state)
assert not torch.allclose(state, reverse_state)
print("Reversed phrase's final card:", [round(value, 3) for value in reverse_state.tolist()])
print("Takeaway: the weights are reused, the card changes, and order can change the result.")


## 2 - A later mistake still has to teach the opening

The card can carry information forward. That is only half the job. If a guess
at the end is wrong, training must work out how the **earlier** uses of the
reader contributed to it.

![Memory passes forward through three uses of one reader; feedback from the final check crosses those state transitions in reverse.](images/01a-03-forward-memory-backward-feedback.png)

*Follow teal reading from left to right, then the orange feedback arrows back.
A longer phrase puts more updates between the final mistake and the first note.*

This is **backpropagation through time**: ordinary backpropagation following
the recorded recurrent steps backward. Each use contributes to the gradients
of the **same shared weights**. The optimizer updates those weights afterward;
it doesn't edit the notes or invent a separately trained reader for each time.

### Make the difficulty small enough to inspect

Imagine a route that passes on just 65% of the feedback at each step.
After two steps it passes on 65% **of the remaining 65%**, not 30% less
than the original amount. Many small losses multiply into a large one.

The next experiment sends a unit feedback signal from the end of a scalar
chain. Autograd measures what reaches its opening. There is no music model
or fitted loss here: 0.65 is a chosen multiplier that isolates this one route.
Real RNN derivatives depend on their weights, inputs, and nonlinearities.

**Predict:** at gaps of 5, 10, and 20 steps, which routes still pass more
than 1%? Check the measured derivatives against repeated multiplication.


In [ ]:
# -- Autograd checks the deliberately simplified feedback route ------------------
distances = np.arange(1, 31)
vanilla_retention = 0.65

def route_gradient(retention, distance):
    opening = torch.tensor(1.0, requires_grad=True)
    carried = opening
    for _ in range(int(distance)):
        carried = retention * carried
    return torch.autograd.grad(carried, opening)[0].item()

vanilla_gradient = np.array([route_gradient(vanilla_retention, gap) for gap in distances])
assert np.allclose(vanilla_gradient, vanilla_retention ** distances)
for distance in (5, 10, 20):
    print(f"distance {distance:2d}: retained gradient = {vanilla_gradient[distance - 1]:.6f}")

first_below_one_percent = int(distances[np.flatnonzero(vanilla_gradient < 0.01)[0]])
print(f"First distance below 1%: {first_below_one_percent} steps")
print("Takeaway: many shrinking transitions can make an early clue hard to teach.")


In [ ]:
# -- Static evidence plus animation: the same recorded decay ---------------------
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(distances, vanilla_gradient, color="#f97316", linewidth=3)
ax.axhline(0.01, color="#ef4444", linestyle="--", label="1% retained")
ax.set(yscale="log", xlabel="Distance from the loss", ylabel="Gradient reaching the cue")
ax.set_title("Chosen scalar route: feedback decays with distance")
ax.legend()
plt.show()

fig, ax = plt.subplots(figsize=(9, 4))
ax.set(xlim=(1, 30), ylim=(1e-6, 1), yscale="log")
ax.set_xlabel("Distance from the loss")
ax.set_ylabel("Gradient reaching the cue")
line, = ax.plot([], [], color="#f97316", linewidth=3)
marker, = ax.plot([], [], "o", color="#ef4444")

def update_decay(frame):
    visible = frame + 1
    line.set_data(distances[:visible], vanilla_gradient[:visible])
    marker.set_data([distances[frame]], [vanilla_gradient[frame]])
    ax.set_title(
        f"Distance {distances[frame]}: {vanilla_gradient[frame]:.6f} retained",
    )
    return line, marker

animation = FuncAnimation(fig, update_decay, frames=len(distances), interval=90, blit=True)
plt.close(fig)
print("Watch the marker: distance grows while the usable teaching signal collapses.")
display(HTML(animation.to_jshtml()))


### That failed, but clipping cannot bring the signal back

A route that repeatedly **shrinks** feedback can make it vanish; one that
repeatedly **amplifies** it can make it explode. **Gradient clipping** puts a
cap on an overly large gradient before the optimizer uses it.

That is a safety brake, not a memory upgrade. Our 20-step route passes about
0.00018 of a unit signal. A cap won't make that tiny signal larger.
[01B's notebook](01B-sequence-memory-pytorch-lab.ipynb) will show exactly where
the brake belongs in the training step.

Read the curve before the animation: a logarithmic axis makes very small
values visible; it doesn't make them useful. The animation replays the same
measured values. It isn't another training run.

**Your turn:** change only the per-step retention. Predict the first gap
below 1%, then compare the printed crossing. What would 1.05 do instead?


In [ ]:
# -- Your turn: change one memory-route variable ---------------------------------
# CHANGE THIS: try 0.50 or 0.85
retention_to_test = 0.75
tested_curve = retention_to_test ** distances
below = np.flatnonzero(tested_curve < 0.01)
crossing = int(distances[below[0]]) if len(below) else None
print(f"Retention {retention_to_test:.2f}: first distance below 1% = {crossing}")
print("Check: a larger retention factor must push the crossing farther away.")


## 3 - Keep old evidence without rewriting all of it

An ordinary RNN makes one fresh card from everything it sees. Keeping an
old clue and reacting to a new note both go through that same rewrite.
Could we keep a separate carried memory, decide how much new content to add,
and choose how much of that memory to expose right now?

That is the useful idea in a **long short-term memory** cell, or **LSTM**.
Forget the acronym for a moment. Read the picture as **keep, write, show**.

![Old memory passes through a keep control, proposed content passes through a write control, and their sum becomes the new carried memory. A separate show control exposes hidden state without replacing the carried memory.](images/01a-04-keep-write-show.png)

*Follow the top lane: old memory has a direct route into new memory.
The lower lane adds only the part of the proposed update we let through.*

| Part | Plain-English job | If its control is near zero |
|---|---|---|
| Forget gate | How much old memory to **keep** | Clear that part of the old memory |
| Input gate | How much proposed content to **write** | Ignore that proposed change |
| Candidate | The **proposed content**, which can be positive or negative | Not a gate: it proposes values |
| Output gate | How much squashed memory to **show** as hidden state | Hide it now, without deleting the carried memory |

A gate is a dimmer, not necessarily an on/off switch. Its values lie between
zero and one. The candidate and memory values need not. Real gates are
learned from the current input and previous exposed hidden state; we don't
write a rule that says which note to remember.

Two rows now travel onward: **cell state** is the carried memory; **hidden
state** is its currently exposed view. The exposed view passes through `tanh`
and the output gate. Changing what is shown needn't erase what is carried.

The next code chooses gates **by hand** so you can inspect the effect
without training noise. It is one memory coordinate, not a complete trained
LSTM. First write a clue, then mostly close the write control.

**Predict:** when we keep most of the old memory and write almost nothing
new, should it vanish immediately or fade slowly? If we close only SHOW,
does the carried memory disappear too?


In [ ]:
# -- Gate evidence: record actual cell-state updates ------------------------------
forget = np.array([0.95, 0.95, 0.90, 0.98, 0.97, 0.96])
input_gate = np.array([0.80, 0.10, 0.05, 0.05, 0.05, 0.05])
candidate = np.array([0.90, -0.20, 0.10, 0.00, 0.05, 0.00])
output = np.array([0.70, 0.75, 0.80, 0.80, 0.85, 0.85])
cell = 0.0
cell_history = []
hidden_history = []

for step in range(len(forget)):
    cell = forget[step] * cell + input_gate[step] * candidate[step]
    hidden = output[step] * np.tanh(cell)
    cell_history.append(cell)
    hidden_history.append(hidden)

for step, (cell_value, hidden_value) in enumerate(zip(cell_history, hidden_history)):
    print(f"step {step}: cell={cell_value:.3f}, visible hidden={hidden_value:.3f}")
hidden_if_show_closed = 0.0 * np.tanh(cell_history[2])
assert hidden_if_show_closed == 0.0 and cell_history[2] != 0.0
assert all(0 <= value <= 1 for gates in (forget, input_gate, output) for value in gates)
print(f"At step 2, close only SHOW: exposed={hidden_if_show_closed:.3f}; carried={cell_history[2]:.3f}")
print("Takeaway: little new writing lets memory persist; hiding it doesn't erase it.")


### Read the trace before pressing play

Each loop pass first keeps part of the old `cell`, then adds the permitted
part of `candidate`. `hidden` is the squashed, exposed view. We save both
rows before moving on. The extra probe changes only the show control:
the exposed value becomes zero while the stored cell stays nonzero.

The static chart below puts the controls above their consequences. After
the first write, the input gate stays low and the memory decays gradually
rather than being replaced. The animation replays these same saved steps.


In [ ]:
# -- Static storyboard and animation: gates control one carried state ------------
steps = np.arange(len(cell_history))
fig, axes = plt.subplots(2, 1, figsize=(10, 7), sharex=True)
axes[0].plot(steps, forget, label="forget", color="#60a5fa", marker="o")
axes[0].plot(steps, input_gate, label="input", color="#f59e0b", marker="o")
axes[0].plot(steps, output, label="output", color="#34d399", marker="o")
axes[0].set_ylabel("Gate value")
axes[0].legend(ncol=3)
axes[1].plot(steps, cell_history, label="cell state", color="#a78bfa", marker="o")
axes[1].plot(steps, hidden_history, label="visible hidden", color="#f472b6", marker="o")
axes[1].set(xlabel="Timestep", ylabel="State value")
axes[1].legend()
plt.show()

fig, ax = plt.subplots(figsize=(9, 4))
ax.set(xlim=(-0.3, len(steps) - 0.7), ylim=(-0.1, 0.9))
ax.set_xlabel("Timestep")
ax.set_ylabel("Recorded state")
cell_line, = ax.plot([], [], color="#a78bfa", linewidth=3, marker="o", label="cell")
hidden_line, = ax.plot([], [], color="#f472b6", linewidth=2, marker="o", label="hidden")
ax.legend()

def update_cell(frame):
    visible = frame + 1
    cell_line.set_data(steps[:visible], cell_history[:visible])
    hidden_line.set_data(steps[:visible], hidden_history[:visible])
    ax.set_title(
        f"f={forget[frame]:.2f}, i={input_gate[frame]:.2f}, o={output[frame]:.2f}",
    )
    return cell_line, hidden_line

gate_animation = FuncAnimation(fig, update_cell, frames=len(steps), interval=550, blit=True)
plt.close(fig)
print("Watch the cell line stay stable while the gates change what is written and exposed.")
display(HTML(gate_animation.to_jshtml()))


## 4 - Does a better route really pass more feedback?

Go back to exactly the same scalar experiment, with the same gaps and unit
feedback at the end. Change only the route multiplier from **0.65 to 0.95**.
The higher value represents a fairly open LSTM keep route with no new writing.

Twenty small losses still matter: keeping 95% at every step is not keeping
100%. But it can leave much more feedback than keeping 65% at every step.

**Predict:** after 20 transitions, will the 0.95 route still pass over a
quarter of the unit feedback? Will the 0.65 route? Autograd checks both.

These are **chosen route settings**, not default multipliers that every RNN
or LSTM has. We haven't trained either architecture here. This isolates why
a controlled carry lane can help; [01C's notebook](01C-cinematic-piano-memory.ipynb)
will test actual fitted models and report whichever result occurs.


In [ ]:
# -- Controlled comparison: same delay, different memory route -------------------
lstm_gradient = np.array([route_gradient(0.95, gap) for gap in distances])
assert np.allclose(lstm_gradient, 0.95 ** distances)
delay = 20
vanilla_at_delay = vanilla_gradient[delay - 1]
lstm_at_delay = lstm_gradient[delay - 1]
ratio = lstm_at_delay / vanilla_at_delay

fig, ax = plt.subplots(figsize=(9, 4))
ax.semilogy(distances, vanilla_gradient, label="vanilla route (0.65)", linewidth=3)
ax.semilogy(distances, lstm_gradient, label="LSTM cell route (0.95)", linewidth=3)
ax.set(xlabel="Distance from loss", ylabel="Gradient reaching cue")
ax.legend()
plt.show()

print(f"At distance {delay}: vanilla={vanilla_at_delay:.6f}, LSTM route={lstm_at_delay:.6f}")
print(f"Controlled retention advantage: {ratio:,.0f}x")
print("Boundary: these are chosen scalar routes, not measured success rates of trained architectures.")
assert lstm_at_delay > vanilla_at_delay


### From a two-number card to real model dimensions

| Our close-up | In a real recurrent model | What stays the same |
|---|---|---|
| Hand-picked two-number note rows | Measured features or trainable embedding rows | The current input is separate from carried state |
| Two-number RNN card | A hidden row, often hundreds of numbers wide | One update rule is reused at every step |
| One hand-controlled LSTM coordinate | Learned vector-valued gates, hidden state, and cell state | Keep, write, and show have different jobs |
| Unit feedback through chosen multipliers | Autograd through the actual weights and nonlinearities | A final error must cross earlier transitions |

**A useful boundary:** gates create a way to keep evidence, not a promise of
perfect memory. The data and training must teach the controls when to use it.
Remembering every distraction would not be an upgrade.

Next, [01B's notebook](01B-sequence-memory-pytorch-lab.ipynb) uses these two
state roles in `nn.LSTM`, first with equal forecast windows and then with
unequal note phrases. **Where should a short phrase's final card come from:
its last real note, or the blank slots we added to fit the batch?**


## 5 - Evidence scorecard and handoff

| Opening question | Evidence | What you can say |
|---|---|---|
| Does counting keep order? | Reversing the notes changes seats, not counts | Pooling removed an available distinction |
| Does recurrence reuse one reader? | Same matrices, changing cards; reversed notes change the final card | State and weights are different objects |
| Can a route lose old feedback? | Autograd agrees with repeated retention; the 0.65 route falls below 1% at 11 steps | Even modest per-step shrinking compounds |
| Does hiding memory erase it? | A zero show control gives zero hidden output with nonzero cell state | Carried and exposed state have separate jobs |
| Do gates guarantee better learning? | Chosen-route comparison only | No trained music-model result has been established here |

| Tier | What we covered |
|---|---|
| Built and measured | Ordered versus pooled inputs, reused recurrence, reversed-input cards, autograd route decay, hand-controlled cell updates, hiding without erasing |
| Explained or illustrated | Forward memory versus backward feedback, three learned gates versus the candidate, clipping's limit, toy-to-real state dimensions |
| Named and out of scope | Packed batches and masked loss (01B), actual RNN/LSTM fitting and listening (01C), direct position-to-position attention (Stage 04) |

**Watch for:** calling the changing card a changing weight; counting notes and
expecting to recover their order; calling the candidate a fourth gate; treating
a hand-chosen retention multiplier as an architecture benchmark.

**Remember:** carry a card forward, send feedback backward, and give useful old
evidence a route that doesn't require rewriting everything at every step.
